In [ ]:
import pandas as pd
import numpy as np

import requests
import os
from dotenv import load_dotenv

import concurrent.futures
import threading
import time


In [ ]:
# load spotify data
df = pd.read_csv(r'excel_data\Alltime_Spotify_Streaming_Data.csv', low_memory=False)

print(f"Loaded Database: {df.shape[0]} rows, {df.shape[1]} columns")


In [ ]:
df

In [ ]:
############################
# STEP 1: HEADERS CLEANING #
############################

df.columns = df.columns.str.replace('_', ' ')

df = df.rename(columns={
    'track album name': 'track artist',
    'track album name2': 'track album name'
})

print(df.columns)


In [ ]:
#############################
# STEP 2: CATEGORICAL TYPOS #
#############################

cleanup_map = {
    'Android OS 5.1.1 API 22 (samsung, SAMSUNG-SM-G920A)': 'SAMSUNG-SM-G920A',
    'iOS 9.2.1 (iPad2,2)': 'iPad2',
    'Android OS 6.0.1 API 23 (samsung, SAMSUNG-SM-G920A)': 'SAMSUNG-SM-G920A',
    'Android OS 7.0 API 24 (samsung, SAMSUNG-SM-G920A)': 'SAMSUNG-SM-G920A',
    'Android-tablet OS 7.0 API 24 (samsung, SAMSUNG-SM-G920A)': 'SAMSUNG-SM-G920A',
    'Android OS 6.0.1 API 23 (HUAWEI, KIW-L24)': 'KIW-L24',
    'Android OS 8.0.0 API 26 (motorola, moto g(6) play)': 'moto g(6) play',
    'Android OS 9 API 28 (motorola, moto g(6) play)': 'moto g(6) play',
    'Android OS 8.0.0 API 26 (samsung, SM-G950U)': 'SM-G950U',
    'Android OS 9 API 28 (samsung, SM-G950U)': 'SM-G950U',
    'Android OS 11 API 30 (samsung, SM-G781U)': 'SM-G781U',
    'Android OS 12 API 31 (samsung, SM-G781U)': 'SM-G781U',
    'not_applicable': np.nan # handle ghost values     
}

df['streaming platform'] = df['streaming platform'].replace(cleanup_map)

print(df['streaming platform'].unique())

In [ ]:
###############################
# STEP 3: DATE TIME DATA TYPE #
###############################

print(df['time stamp'].dtype)

df['time stamp'] = pd.to_datetime(df['time stamp'],errors = 'coerce')

df['time stamp']

In [ ]:
###################################
# STEP 4: LOGICAL INTEGRITY CHECK #
#      A: (No Start or End)       #
###################################

impossible_track1 = df['reason start'].isna()
impossible_track2 = df['reason end'].isna()

print(df.loc[impossible_track1])
print(df.loc[impossible_track2])

In [ ]:
###################################
# STEP 4: LOGICAL INTEGRITY CHECK #
#      b: (less Than 0 ms Played) #
###################################
# step 4b: check logical integrity (less than 0 ms played)
impossible_track3 = df['ms played'] < 0

print(df.loc[impossible_track3])

In [ ]:
#######################################
# STEP 5: REMOVE UNNECESSARY ROWS     #
#      A: (Tracks that are not songs) #
#######################################

# check if there are podcast or audiobook tracks
not_songs = df['episode name'].notna() | df['audiobook title'].notna()
# print(df.loc[not_songs])

# remove these tracks from dataframe
df = df[~not_songs]

df

In [ ]:
#######################################
# STEP 5: REMOVE UNNECESSARY ROWS     #
#      B: (Tracks played for 0 ms)    #
#######################################

# check if there are tracks played for 0 seconds
zero_ms = df['ms played'] == 0
# print(df.loc[zero_ms])

# remove these tracks from dataframe
df = df[~zero_ms]

df

In [ ]:
######################################
# STEP 6: REMOVE UNNECESSARY COLUMNS #
######################################

df = df.drop(['episode name', 'episode show name', 'episode spotify url', 
              'audiobook title', 'audiobook url', 'audiobook chapter url',
              'audiobook chapter title', 'offline timestamp'],axis=1)

df

In [ ]:
######################################
# STEP 7: INDENTIFY UNIQUE TRACKS    #
######################################

df_unique = df.drop_duplicates(subset=['track spotify url'])
df_unique = df_unique[['track title', 'track artist', 'track spotify url']]

# determine number of unique songs
num_unique_songs = df_unique.shape[0]
user_message = f"You have listened to {num_unique_songs} different tracks in the last 10 years!"
print(user_message)

# save list of unique songs as csv file
df_unique.to_csv('unique_songs.csv', index=False)

In [ ]:
#########################################
# STEP 8: CLEAN CSV TO UPLOAD NEW DATA  #
#########################################


df2 = pd.read_csv('unique_songs.csv')
df2['spotify track id'] = df2['track spotify url'].str.split(':', n=2).str[-1]

# create new columns for api call variables
added_columns = [
    "id", "key", "mode", "camelot", "tempo", "duration",
    "popularity", "energy", "danceability", "happiness",
    "acousticness", "instrumentalness", "liveness",
    "speechiness", "loudness"
]

for col in added_columns:
    df2[col] = None

df2


In [ ]:
#########################################
# STEP 9a: SET UP FOR GET API FUNCTION  #
#########################################
load_dotenv()
api_key = os.getenv("RAPIDAPI_KEY")

# initialize lock
lock = threading.Lock()
        
# set up api call session
headers = {
        'x-rapidapi-key': api_key,
        'x-rapidapi-host': 'track-analysis.p.rapidapi.com'
    }
session = requests.Session()
session.headers.update(headers)

# keep track of failed track requests
failed_tracks = []

In [ ]:
#########################################
# STEP 9b: DEFINE GET API FUNCTION      #
#########################################
def get_api_data(i):
    # set spotifyId based on index i
    spotifyId = df2.at[i,'spotify track id']
    
    # configure api request
    url = f"https://track-analysis.p.rapidapi.com/pktx/spotify/{spotifyId}"
    
    try:
        # make api call
        result = session.get(url, timeout=5)
        # error check
        if result.status_code == 200:
            output = result.json()
        else:
            output = {}
            print(f"Error {result.status_code}: \n{result.text}")
            with lock:
                failed_tracks.append(i)
    except requests.exceptions.RequestException as ex:
        output = {}
        print(f"API Call failed for track {i}: {ex}")
        with lock:
            failed_tracks.append(i)

    # thread safety to avoid corrupt data
    with lock:
        for col in added_columns:
            df2.at[i,col] = output.get(col)
    
    # buffer for rate limit
    time.sleep(1.5 + random.random())
            
    # debugging setup
    return i

In [ ]:
#########################################
# STEP 9c: ITERATE API CALLS IN BATCHES #
#########################################

# batch variables
batch_count= 1
batch_size = 100
total_tracks = 27434

batch_start = 1600
batch_end = batch_start + batch_size

# save header to csv file, before batch runs
# df2.iloc[0:0].to_csv('progress_backup2.csv', mode='w', header=True, index=False )

# run get_api_data for all tracks, 1 batch (100 api calls) at a time
while True:
    for task in range(batch_start, batch_end):
        print(f"Starting Batch {batch_count}: {batch_start} - {batch_start+batch_size}.")
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
            futures = [executor.submit(get_api_data, i) for i in range(batch_start,batch_end)]
            
            for count, task in enumerate(concurrent.futures.as_completed(futures)):
                task.result()
        
        # save this batch's rows into csv file
        df2.iloc[batch_start:batch_end].to_csv('progress_backup2.csv', mode='a', header=False, index=False )
        
        # while break & increment
        batch_start += batch_size
        
        if (batch_end + batch_size) > total_tracks:
            batch_end = total_tracks
        elif batch_end == total_tracks:
            break
        else:
            batch_end = batch_start + batch_size
        
        batch_count += 1
    break

# signal end of api calls
print("Finished processing all tracks!")
print(f"{len(failed_tracks)} failed.")
print(failed_tracks)

# read successfully till 1000 rows, then started showing subsequent run time errors
